# RENDU INFO TC1

## Objectif du devoir

Le but de ce devoir est de **déterminer automatiquement une palette de couleurs optimale** pour une image donnée. Cette palette devra valider les contraintes suivantes : 

1. utiliser moins de couleurs que le nombre disponible dans l'image donnée;
2. être la plus représentative possible des couleurs de l'image donnée. 

Comme nous l'avons vu dans le TD 4, les couleurs peuvent être encodée par composantes rouge, verte et bleue (soit 256 valeurs possibles par composante, autrement dit sur 8 bits) ainsi potentiellement utiliser $256 \times 256 \times 256 = 16 777 216$ couleurs. En réalité, beaucoup moins sont nécessaires et surtout perceptibles par l'humain. Réduire le nombre de couleurs ou réaliser une "_quantification de couleurs_" est une tâche fréquente et c'est une fonctionnalité classique des outils éditeurs d'images (Photoshop, Gimp, etc.) implémentée aussi dans le module Pillow de Python. A noter que cette réduction s'effectue avec perte de couleurs et doit être réalisée avec les bons paramètres (nombre et choix des couleurs) ce qui est votre objectif. 

La figure ci-dessous illustre le problème à résoudre : étant donnée une image en entrée, proposer une liste de couleurs (que l'on appellera la palette), afin de re-colorier une image en sortie.

<div style="text-align:center;">
<table>
  <tr>
    <td>
      <img src="figures/color-rainbow.png" alt="Image originale" style="height:5cm;">
      <p>Image donnée</p>
    </td>
    <td>
      <img src="figures/rainbow-palette-8.png" alt="Palette de 8 couleurs représentatives" style="height:5cm;">
      <p>Palette de 8 couleurs représentatives</p>
    </td>
    <td>
      <img src="figures/rainbow-recoloriee.png" alt="Image originale recoloriée avec la palette" style="height:5cm;">
      <p>Image donnée recoloriée avec la palette</p>
    </td>
  </tr>
</table>
</div>

## Étapes de travail

Voici des étapes de travail suggérées :

1. Prenez une image de votre choix (pas trop grande) en la chargeant avec PIL. Lister les couleurs présentes, identifier celles qui sont uniques et leur fréquence.

In [ ]:
from PIL import Image
from IPython.display import display


def setRegion(x, y, w, h, color, px):
    for i in range(int(x), int(x + w)):
        for j in range(int(y), int(y + h)):
            px[i, j] = color

def lister_couleurs(im):
    """
    Retourne un dictionnaire {couleur: fréquence} pour tous les pixels de l'image.
    Arguments:
        im (Image): image PIL en mode RGB
    Renvoie :
        dict: {(r,g,b): compteur}
    """
    px = im.load()
    W, H = im.size
    freq = {}
    for x in range(W):
        for y in range(H):
            c = px[x, y]
            if c in freq:
                freq[c] += 1
            else:
                freq[c] = 1
    return freq

In [ ]:
# Test sur une image simple : image noire avec un carré blanc
W, H = 100, 100
im_test = Image.new('RGB', (W, H))
px_test = im_test.load()
setRegion(W//3, H//3, W//3, H//3, (255, 255, 255), px_test)
display(im_test)

freq_test = lister_couleurs(im_test)
print("Nombre de couleurs :", len(freq_test))
liste_couples = []
for couleur, nombre in freq_test.items():
    liste_couples.append((nombre, couleur))

liste_couples.sort(reverse=True)

for n, c in liste_couples:
    print(f"{c} : {n} pixels")

assert len(freq_test) == 2, "Il doit y avoir exactement 2 couleurs"

In [ ]:
# Test sur une image plus complexe
im = Image.open("lyon.png").convert("RGB")
freq = lister_couleurs(im)
print(f"Taille image : {im.size}")
print(f"Nombre de couleurs : {len(freq)}")
liste_couples = []
for couleur, nombre in freq.items():
    liste_couples.append((nombre, couleur))

liste_couples.sort(reverse=True)
top10 = liste_couples[:10]
print("Top 10 couleurs les plus fréquentes :")
for n, c in top10:
    print(f"{c} : {n} pixels")
display(im)

2. Proposez une méthode (naïve pour commencer) de choix d'une palette de $k$ couleurs. Affichez là sous forme d'image (exemple de d'image au milieu de la figure du dessus) avec une nouvelle image PIL. Utilisez également des images simples où le résultat attendu est connu comme pour les images ci-dessous :

  <div style="text-align:center;">
    <table>
      <tr>
        <td>
          <img src="figures/1-color-back.png" alt="1 couleur noir" style="width:3cm;">
          <p>1 couleur noir</p>
        </td>
        <td>
          <img src="figures/4-color.png" alt="4 couleurs" style="width:3cm;">
          <p>4 couleurs</p>
        </td>
      </tr>
    </table>
  </div>

In [ ]:
# 2. PALETTE NAÏVE (On prend les k couleurs les plus fréquentes)

def palette_naive(freq, k):
    """
    Retourne une palette de k couleurs = les k couleurs les plus fréquentes.
    Arguments :
        freq (dict): dictionnaire {couleur: fréquence}
        k (int): nombre de couleurs souhaitées dans la palette
    Renvoie :
        list: liste de k tuples (r, g, b) (palette)
    """
    # Trier par fréquence décroissante et prendre les k premières
    liste_couples = []
    for couleur, freq in freq.items():
        liste_couples.append((freq, couleur))

    liste_couples.sort(reverse=True)
    k_couples = liste_couples[:k]
    return [c for c, _ in k_couples]

In [ ]:
def afficher_palette(palette, taille_case=50):
    """
    Crée et affiche une image PIL montrant les couleurs de la palette.
    Args:
        palette (list): liste de tuples (r, g, b)
        taille_case (int): taille en pixels de chaque case de couleur
    Returns:
        Image: image PIL de la palette
    """
    k = len(palette)
    im_palette = Image.new('RGB', (k * taille_case, taille_case))
    px_p = im_palette.load()
    for i in range(len(palette)):
        couleur = palette[i]
        setRegion(i * taille_case, 0, taille_case, taille_case, couleur, px_p)
    return im_palette

# Test sur image 1 couleur (noire)
im_1 = Image.new('RGB', (50, 50))  # image entièrement noire
freq_1 = lister_couleurs(im_1)
palette_1 = palette_naive(freq_1, 1)
print("Palette image noire (k=1):", palette_1)
assert palette_1 == [(0, 0, 0)], "Doit retourner [(0,0,0)]"
display(afficher_palette(palette_1))

# Test sur image 4 couleurs (rouge, vert, bleu, jaune)
im_4 = Image.new('RGB', (100, 100))
px_4 = im_4.load()
setRegion(0, 0, 50, 50, (255, 0, 0), px_4) # rouge
setRegion(50, 0, 50, 50, (0, 255, 0), px_4) # vert
setRegion(0, 50, 50, 50, (0, 0, 255), px_4) # bleu
setRegion(50, 50, 50, 50, (255, 255, 0), px_4) # jaune

freq_4 = lister_couleurs(im_4)
palette_4 = palette_naive(freq_4, 4)
print("Palette image 4 couleurs (k=4):", sorted(palette_4))
assert len(palette_4) == 4
display(afficher_palette(palette_4))
display(im_4)

# Test sur une image plus complexe
im = Image.open("lyon.png").convert("RGB")
freq = lister_couleurs(im)
palette = palette_naive(freq, 8)
print("Palette image 8 couleurs (k=8):", sorted(palette))
assert len(palette) == 8
display(afficher_palette(palette))
display(im)

3. Re-coloriez une image avec une palette de $k$ couleurs, et affichez le résultat sous forme d'image PIL. Pour re-colorier chaque pixel, prendre la couleur la plus proche dans la palette en utilisant une fonction de distance (Euclidienne par exemple..).

In [ ]:
# 3. RE-COLORIER avec palette
from math import sqrt
def distance(c1, c2):
    return sqrt((c1[0]-c2[0])**2 + (c1[1]-c2[1])**2 + (c1[2]-c2[2])**2)

def couleur_la_plus_proche(c, palette):
    """
    Retourne la couleur de la palette la plus proche de c (distance euclidienne).
    Args:
        c (tuple): couleur (r, g, b)
        palette (list): liste de tuples (r, g, b)
    Returns:
        tuple: couleur la plus proche dans la palette
    """
    plus_proche = palette[0]
    dist_min = distance(c, palette[0])
    for couleur in palette[1:]:
        d = distance(c, couleur)
        if d < dist_min:
            dist_min = d
            plus_proche = couleur
    return plus_proche

def recolorier(im, palette):
    """
    Re-colorier chaque pixel de l'image avec la couleur la plus proche dans la palette.
    Arguments :
        im (Image): image PIL originale
        palette (list): liste de tuples (r, g, b)
    Renvoie :
        Image: nouvelle image re-coloriée
    """
    W, H = im.size
    im_out = Image.new('RGB', (W, H))
    px_in = im.load()
    px_out = im_out.load()
    for x in range(W):
        for y in range(H):
            c = px_in[x, y]
            c_proche = couleur_la_plus_proche(c, palette)
            px_out[x, y] = c_proche
    return im_out

In [ ]:

# --- Test : image 4 couleurs re-coloriée avec palette de 2 couleurs ---
# On prend rouge et bleu comme palette réduire → vert et jaune iront vers le plus proche
palette_reduite = [(255, 0, 0), (0, 0, 255)]
im_recoloriee = recolorier(im_4, palette_reduite)
print("Re-coloriage avec 2 couleurs (rouge, bleu) :")
display(im_4)         # originale
display(im_recoloriee) # re-coloriée

# --- Test : image noire re-coloriée avec palette de 1 couleur ---
im_1_recoloriee = recolorier(im_1, [(0, 0, 0)])
freq_1r = lister_couleurs(im_1_recoloriee)
assert list(freq_1r.keys()) == [(0, 0, 0)], "L'image doit être entièrement noire"
print("✓ Test image 1 couleur réussi")

# --- Test complet : image 4 couleurs re-coloriée avec palette exacte ---
im_4_exact = recolorier(im_4, palette_4)
freq_exact = lister_couleurs(im_4_exact)
assert len(freq_exact) == 4, "Doit avoir exactement 4 couleurs"
print("✓ Test image 4 couleurs avec palette exacte réussi")

La palette ne représente pas l'ensemble des couleurs : elle ne comprend que les couleurs les plus présentes, qui sont toutes concentrées dans le ciel.


4. Proposez une méthode de validation de votre approche. Par exemple affichez la différence entre l'image originale et celle re-coloriée. Calculez un score global d'erreur.


In [ ]:
# 4. VALIDATION
def calcul_erreur(im1, im2):
    px1, px2 = im1.load(), im2.load()
    err = 0
    for x in range(W):
        for y in range(H):
            err += distance(px1[x, y], px2[x, y])
    return err / (W * H)

erreur_palette_naive = calcul_erreur(im, im_recolor_naive)
print(f"Erreur de la palette naïve : {erreur_palette_naive:.2f} (moyenne de la distance euclidienne des couleurs)")


5. Améliorez le choix des $k$ couleurs afin de minimiser l'erreur entre l'image originale et re-coloriée. Une piste possible est de trier les couleurs dans une liste, diviser cette liste en $k$ intervals de couleurs et prendre la couleur du milieu de chaque interval. D'autres méthodes plus avancées peuvent être explorées !


In [ ]:
# 5. AMÉLIORATION (Choix par intervalles de luminosité)
palette_opti = []
# Tri des couleurs auxquelles on est les plus sensibles 
liste_couples_score = []
for c in couleurs:
    score = 0.3 * c[0] + 0.59 * c[1] + 0.11 * c[2]
    liste_couples_score.append((score, c))

liste_couples_score.sort()

couleurs_triees = [] #Liste des couleurs triées selon notre sensibilité (0=plus sensible)
for score, c in liste_couples_score:
    couleurs_triees.append(c)

n = len(couleurs)
for i in range(k):
    milieu_paquet = int((i + 0.5) * n / k)
    palette_opti.append(couleurs[milieu_paquet])

im_tri = appliquer_palette(im, palette_opti)
print("Image re-coloriée (Optimisée par intervalles) :")
display(im_tri)

erreur_palette_tri = calcul_erreur(im, im_tri)
print(f"Erreur Optimisée : {erreur_palette_tri:.2f} (moyenne de la distance euclidienne des couleurs)")

In [ ]:
#Autre méthode : K-means

def calculer_moyenne(groupe):
    somme_r = 0
    somme_g = 0
    somme_b = 0
    for c in groupe: # c est un tuple (R, G, B)
        somme_r += c[0]
        somme_g += c[1]
        somme_b += c[2]
        
    n = len(groupe)
    return (somme_r // n, somme_g // n, somme_b // n)

def methode_kmeans(couleurs_uniques, k, iterations=5):
    n = len(couleurs_uniques)
    # Initialisation : on prend des couleurs réparties dans la liste
    centres = [couleurs_uniques[i * n // k] for i in range(k)]
    
    for _ in range(iterations):
        groupes = [[] for _ in range(k)]
        
        # --- ÉTAPE D'ASSIGNATION ---
        for c in couleurs_uniques:
            index_proche = 0
            dist_min = distance(c, centres[0])
            
            for i in range(1, k):
                d = distance(c, centres[i])
                if d < dist_min:
                    dist_min = d
                    index_proche = i
            
            groupes[index_proche].append(c)
            
        # --- ÉTAPE DE MISE À JOUR ---
        nouveaux_centres = []
        for i in range(k):
            if groupes[i]:
                nouveaux_centres.append(calculer_moyenne(groupes[i]))
            else:
                # Sécurité si groupe vide
                nouveaux_centres.append(couleurs_uniques[i * n // k])
        
        centres = nouveaux_centres
                
    return centres

# Utilisation
palette_kmeans = methode_kmeans(couleurs, 8)
im_kmeans = appliquer_palette(im, palette_kmeans)
print("Image re-coloriée (Optimisée par k-means) :")
display(im_kmeans)

erreur_palette_kmeans = calcul_erreur(im, im_kmeans)
print(f"Erreur Optimisée : {erreur_palette_kmeans:.2f} (moyenne de la distance euclidienne des couleurs)")


6. Testez votre palette sur plusieurs images de votre choix ou générées automatiquement avec un nombre et une distribution connue de couleurs. Comparer les performances de vos techniques avec d'autres méthodes (cette fois vous pouvez utiliser un éditeur de texte ou la fonction _quantize_ de PIL [(doc)](https://pillow.readthedocs.io/en/stable/reference/Image.html).


In [ ]:
# --- 1. Charger l'image de test ---
im_test = Image.open("lyon.png").convert("RGB")
k = 8 # On garde la même contrainte de 8 couleurs

# --- 3. Méthode Pillow (quantize) ---
# La méthode quantize renvoie une image en mode "P" (Palette)
# On la convertit en "RGB" pour pouvoir comparer les pixels à la fin
im_pillow = im_test.quantize(colors=k).convert("RGB")

display(im_pillow)

erreur_palette_pillow = calcul_erreur(im, im_pillow)
print(f"Erreur Pillow : {erreur_palette_pillow:.2f} (moyenne de la distance euclidienne des couleurs)")

In [ ]:
# --- 4. Affichage des résultats ---
print("--- ANALYSE DES PERFORMANCES ---")
print(f"Erreur de la méthode 1 : {erreur_palette_tri:.2f}")
print(f"Erreur de la méthode 2 : {erreur_palette_kmeans:.2f}")
print(f"Erreur de la méthode Pillow : {erreur_palette_pillow:.2f}")

# Conclusion
def diff(erreur_palette):
    return ((erreur_palette - erreur_palette_pillow) / erreur_palette) * 100
print(f"Pillow est {diff(erreur_palette_tri):.1f}% plus précis que la méthode 1.")
print(f"Pillow est {diff(erreur_palette_kmeans):.1f}% plus précis que la méthode 2.")


7. Utilisez un pré-traitement des images (flou gaussien, etc) afin de lisser les couleurs. Cela est une piste afin de choisir de meilleurs couleurs représentatives. Proposez une comparaison de cette amélioration (ou de déterioration éventuelle) avec les autres méthodes.


In [ ]:
from PIL import Image, ImageFile, ImageDraw, ImageChops, ImageFilter
im_floue = im.filter(ImageFilter.GaussianBlur)

print("Image après pré-traitement (Flou Gaussien) :")
display(im_floue)

In [ ]:
# Statistiques de l'image floue
stats_flou = {}
W, H = im_floue.size
px_flou = im_floue.load()

for x in range(W):
    for y in range(H):
        c = px_flou[x, y]
        stats_flou[c] = stats_flou.get(c, 0) + 1

# Générer la palette à partir des couleurs lissées
couleurs_uniques_flou = list(stats_flou.keys())
palette_lissee = methode_kmeans(couleurs_uniques_flou, k)

In [ ]:
# A. Application de la palette normale (générée sur l'image nette)
im_standard = appliquer_palette(im, palette_kmeans)

# B. Application de la palette lissée (générée sur l'image floue)
im_avec_pre_traitement = appliquer_palette(im, palette_lissee)

print("Résultat SANS pré-traitement :")
display(im_standard)

print("Résultat AVEC pré-traitement (Flou) :")
display(im_avec_pre_traitement)


8. Proposez une méthode d'amélioration de calcul de la distance entre deux couleurs, vous pouvez vous baser sur d'autres espaces de couleur [(doc)](https://fr.wikipedia.org/wiki/Espace_de_couleur). Cette partie est difficile, les espaces de couleurs possibles sont complexes à comprendre.



9. Optimisez les étapes précédentes (complexité, espace nécessaire, structures de données, etc.) et justifiez vos choix.


Cache, voir autre code

### Bonus

10. Créez une palette représentative à partir de plusieurs images.